# 02 — Feature Engineering

Construction of model predictors from the synthetic patient JSON.

Features include BMI, BSA, indexed EOA, PPM category, gradient progression and longitudinal echo-derived variables.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_FILE = Path('../data/processed/synthetic_2000_patients.json')
SEED = 20260916

with DATA_FILE.open('r', encoding='utf-8') as f:
    patients = json.load(f)

print(f'Loaded {len(patients)} patients')

In [ ]:
def number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

def first_item(value):
    if isinstance(value, list):
        return value[0] if value else {}
    if isinstance(value, dict):
        return value
    return {}

In [ ]:
def ppm_category(ieoa, bmi):
    if np.isnan(ieoa):
        return 'unknown'

    if np.isnan(bmi):
        if ieoa <= 0.65:
            return 'severe'
        if ieoa <= 0.85:
            return 'moderate'
        return 'none'

    if bmi < 30:
        if ieoa <= 0.65:
            return 'severe'
        if ieoa <= 0.85:
            return 'moderate'
        return 'none'

    if ieoa <= 0.55:
        return 'severe'
    if ieoa <= 0.70:
        return 'moderate'
    return 'none'

In [ ]:
rows = []

for patient in patients:
    demographics = patient.get('demographics_and_body_size', {})
    procedures = first_item(patient.get('aortic_valve_procedures', []))
    echoes = patient.get('echocardiograms', [])

    if isinstance(echoes, dict):
        echoes = [echoes]

    reference = first_item(echoes)
    latest = first_item(echoes[-1:])

    age = number(demographics.get('age'))
    bmi = number(demographics.get('bmi'))
    height = number(demographics.get('height_cm'))
    weight = number(demographics.get('weight_kg'))

    if np.isnan(bmi) and not np.isnan(height) and not np.isnan(weight):
        height_m = height / 100
        if height_m > 0:
            bmi = weight / height_m ** 2

    bsa = np.nan
    if not np.isnan(height) and not np.isnan(weight):
        bsa = np.sqrt(height * weight / 3600)

    eoa = number(reference.get('eoa'))
    ieoa = eoa / bsa if not np.isnan(eoa) and bsa > 0 else np.nan

    ref_gradient = number(reference.get('mean_gradient'))
    latest_gradient = number(latest.get('mean_gradient'))
    ref_dvi = number(reference.get('dvi'))
    latest_dvi = number(latest.get('dvi'))
    ref_eoa = number(reference.get('eoa'))
    latest_eoa = number(latest.get('eoa'))

    gradient_change = latest_gradient - ref_gradient
    if np.isnan(ref_gradient) or np.isnan(latest_gradient):
        gradient_change = np.nan

    gradient_change_pct = np.nan
    if not np.isnan(ref_gradient) and ref_gradient != 0 and not np.isnan(latest_gradient):
        gradient_change_pct = gradient_change / ref_gradient * 100

    eoa_change_pct = np.nan
    if not np.isnan(ref_eoa) and ref_eoa != 0 and not np.isnan(latest_eoa):
        eoa_change_pct = (latest_eoa - ref_eoa) / ref_eoa * 100

    dvi_change_pct = np.nan
    if not np.isnan(ref_dvi) and ref_dvi != 0 and not np.isnan(latest_dvi):
        dvi_change_pct = (latest_dvi - ref_dvi) / ref_dvi * 100

    rows.append({
        'patient_id': patient.get('patient_id'),
        'age': age,
        'bmi': bmi,
        'bsa': bsa,
        'reference_mean_gradient': ref_gradient,
        'reference_eoa': ref_eoa,
        'reference_ieoa': ieoa,
        'reference_dvi': ref_dvi,
        'gradient_change': gradient_change,
        'gradient_change_pct': gradient_change_pct,
        'eoa_change_pct': eoa_change_pct,
        'dvi_change_pct': dvi_change_pct,
        'prior_echo_count': len(echoes),
        'ppm_category': ppm_category(ieoa, bmi),
    })

features = pd.DataFrame(rows)
features.head()

In [ ]:
features.describe().T

In [ ]:
features['ppm_category'].value_counts(dropna=False)

## Leakage control

Only predictors available at or before the prediction landmark should enter a model. Event-defining information after the landmark must not be used as a predictor.